# A Universal Prompt for Feature Discovery

In [1]:
import os
import requests
import time
import datetime
import json
import threading
import pandas as pd

## Prompt

The prompt assumes the following metadata is provided:

* Dataset Name
* Description
* Target and its values
* Example of random forty rows

In [3]:
from string import Template

prompt_template = Template("""
{
    "system_message": "IMPORTANT: Return only a valid JSON object with no explanations, text, or markdown!!! Do not include any commentary or introductory text!!!",
    "input_metadata": {
        "dataset_name": "$name",
        "description": "$description",
        "target": "$target",
        "examples": "$examples"
    },
    "task": {
        "steps": [
            "Analyze the provided metadata and examples to determine the domain and context of the dataset.",
            "Identify the key characteristics of the dataset relevant to predicting the target variable.",
            "List potential high-level categorical and numerical features based on domain knowledge inferred from the dataset description.",
            "Extract additional potential features from dataset examples using syntactic and semantic patterns, ensuring at least 20 distinct features are generated.",
            "If the text implies certain values that match the target, these values may also be extracted as features. In cases where the target has multiple values, each value can be independently derived from the text as a feature if it is contextually appropriate.",
            "For text-based datasets, identify key phrases, structural components, and linguistic patterns that are relevant.",
            "For numerical datasets, identify aggregation patterns, distributional characteristics, and possible transformations.",
            "Group related features into meaningful categories where applicable.",
            "If a feature has more than 15 unique categories, group less frequent categories into an 'Other' class.",
            "For each identified feature, provide a clear name, description, a complete list of possible values, and a specific LLM extraction query."
        ],
        "constraints": [
            "Ensure features are distinct and non-redundant.",
            "Note that the target variable is not explicitly present in the input text.",
            "Prioritize domain-specific insights over generic ones.",
            "Ensure output is a structured, valid JSON format.",
            "For categorical variables, list possible values with domain justification.",
            "For numeric variables, provide possible transformations (e.g., log, mean differences).",
            "The extraction queries must be specific and detailed to ensure high-quality feature generation.",
            "Tailor extraction queries to the domain context of the dataset.",
            "Generate a diverse set of features to maximize potential predictive power."
        ]
    },
    "output_format": {
        "type": "json",
        "structure": {
            "features": [
                {
                    "feature_name": "<Name of the categorical or numerical feature>",
                    "description": "<Short description of what the feature represents and how it relates to the dataset's context>",
                    "possible_values": ["<Value 1>", "<Value 2>", "...", "<Value n>"],
                    "extraction_query": "Identify the '<feature_name>' based on the provided context. Options: '<Value 1>', '<Value 2>', ..., '<Value n>'."
                }
            ]
        }
    }
}
""")

In [4]:
# Generate prompt
def generate_prompt(name, description, target, examples_df, text, target_col):
    # 40 random rows
    random_examples = examples_df[[text, target_col]].sample(n=40, random_state=42).to_dict(orient='records')
    examples = '\n'.join([str(row) for row in random_examples])
    
    return prompt_template.substitute(name=name, description=description, target=target, examples=examples)

## Datasets

### C19

In [ ]:
c19_dataset = pd.read_csv("c19_final.csv")[['abstract', 'cited']]
c19_dataset

### M17

In [ ]:
m17_dataset = pd.read_csv("m17_final.csv")[['abstract', 'evaluation']]
m17_dataset

### Hate

In [5]:
# https://paperswithcode.com/dataset/hate-speech
from datasets import load_dataset

dataset = load_dataset("odegiber/hate_speech18")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'user_id', 'subforum_id', 'num_contexts', 'label'],
        num_rows: 10944
    })
})

In [6]:
import pandas as pd
hate_dataset = pd.DataFrame(dataset['train'])
hate_dataset

,text,user_id,subforum_id,num_contexts,label
0,"As of March 13th , 2014 , the booklet had been...",572066,1346,0,0
1,In order to help increase the booklets downloa...,572066,1346,0,0
2,( Simply copy and paste the following text int...,572066,1346,0,0
3,Click below for a FREE download of a colorfull...,572066,1346,0,1
4,Click on the `` DOWNLOAD ( 7.42 MB ) '' green ...,572066,1346,0,0
...,...,...,...,...,...
10939,"Billy - `` That guy would n't leave me alone ,...",734541,1388,0,0
10940,Wish we at least had a Marine Le Pen to vote f...,735154,1388,0,0
10941,Its like the choices are white genocide candid...,735154,1388,0,0
10942,Why White people used to say that sex was a si...,572266,1388,0,1


### BANKING77

In [7]:
# https://paperswithcode.com/dataset/banking77-oos
from datasets import load_dataset

data = load_dataset("banking77")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [8]:
import pandas as pd
bank_dataset = pd.DataFrame(data['train'])
bank_dataset

,text,label
0,I am still waiting on my card?,11
1,What can I do if my card still hasn't arrived ...,11
2,I have been waiting over a week. Is the card s...,11
3,Can I track my card while it is in the process...,11
4,"How do I know if I will get my card, or if it ...",11
...,...,...
9998,You provide support in what countries?,24
9999,What countries are you supporting?,24
10000,What countries are getting support?,24
10001,Are cards available in the EU?,24


In [9]:
bank_dataset_test = pd.DataFrame(data['test'])
bank_dataset_test

,text,label
0,How do I locate my card?,11
1,"I still have not received my new card, I order...",11
2,I ordered a card but it has not arrived. Help ...,11
3,Is there a way to know when my card will arrive?,11
4,My card has not arrived yet.,11
...,...,...
3075,"If i'm not in the UK, can I still get a card?",24
3076,How many countries do you support?,24
3077,What countries do you do business in?,24
3078,What are the countries you operate in.,24


In [10]:
bank_dataset = pd.concat([bank_dataset, bank_dataset_test], axis=0)

## List of All Datasets with Description

In [15]:
datasets = [
        {
        'df': c19_dataset,
        'prefix': 'c19',
        'name': 'CORD-19 scientific articles',
        'description': 'Scientific articles from multiple disciplines and a target being a proxy for research impact.',
        'text_column': 'abstract',
        'target_column': 'cited',
        'target': 'cited (0/1). Target is a binary variable - with 1 representing works cited more than the median value adjusted in time, 0 represents articles with lower or equal to the median citation count.',
        },
        {
        'df': m17_dataset,
        'prefix': 'm17',
        'name': 'M17+ scientific articles',
        'description': 'Scientific articles from multiple disciplines and a target being a proxy for research impact.',
        'text_column': 'abstract',
        'target_column': 'evaluation',
        'target': 'evaluation 1 to 5. Target is an ordinal variable of human evaluation of the quality of articles, 1 indicates works of world-class quality, 5 means mediocre articles of low importance and impact.',
        },
        {
        'df': bank_dataset,
        'prefix': 'bank77',
        'name': 'BANKING77 dataset',
        'description': 'BANKING77 dataset provides a very fine-grained set of intents in a banking domain. It comprises 13,083 customer service queries labeled with 77 intents. It focuses on fine-grained single-domain intent detection.',
        'text_column': 'text',
        'target_column': 'label',
        'target': """
                   "label": datasets.features.ClassLabel(
                    names=[
                        "activate_my_card",
                        "age_limit",
                        "apple_pay_or_google_pay",
                        "atm_support",
                        "automatic_top_up",
                        "balance_not_updated_after_bank_transfer",
                        "balance_not_updated_after_cheque_or_cash_deposit",
                        "beneficiary_not_allowed",
                        "cancel_transfer",
                        "card_about_to_expire",
                        "card_acceptance",
                        "card_arrival",
                        "card_delivery_estimate",
                        "card_linking",
                        "card_not_working",
                        "card_payment_fee_charged",
                        "card_payment_not_recognised",
                        "card_payment_wrong_exchange_rate",
                        "card_swallowed",
                        "cash_withdrawal_charge",
                        "cash_withdrawal_not_recognised",
                        "change_pin",
                        "compromised_card",
                        "contactless_not_working",
                        "country_support",
                        "declined_card_payment",
                        "declined_cash_withdrawal",
                        "declined_transfer",
                        "direct_debit_payment_not_recognised",
                        "disposable_card_limits",
                        "edit_personal_details",
                        "exchange_charge",
                        "exchange_rate",
                        "exchange_via_app",
                        "extra_charge_on_statement",
                        "failed_transfer",
                        "fiat_currency_support",
                        "get_disposable_virtual_card",
                        "get_physical_card",
                        "getting_spare_card",
                        "getting_virtual_card",
                        "lost_or_stolen_card",
                        "lost_or_stolen_phone",
                        "order_physical_card",
                        "passcode_forgotten",
                        "pending_card_payment",
                        "pending_cash_withdrawal",
                        "pending_top_up",
                        "pending_transfer",
                        "pin_blocked",
                        "receiving_money",
                        "Refund_not_showing_up",
                        "request_refund",
                        "reverted_card_payment?",
                        "supported_cards_and_currencies",
                        "terminate_account",
                        "top_up_by_bank_transfer_charge",
                        "top_up_by_card_charge",
                        "top_up_by_cash_or_cheque",
                        "top_up_failed",
                        "top_up_limits",
                        "top_up_reverted",
                        "topping_up_by_card",
                        "transaction_charged_twice",
                        "transfer_fee_charged",
                        "transfer_into_account",
                        "transfer_not_received_by_recipient",
                        "transfer_timing",
                        "unable_to_verify_identity",
                        "verify_my_identity",
                        "verify_source_of_funds",
                        "verify_top_up",
                        "virtual_card_not_working",
                        "visa_or_mastercard",
                        "why_verify_identity",
                        "wrong_amount_of_cash_received",
                        "wrong_exchange_rate_for_cash_withdrawal",
                    ]        
""",  
    },
    {
        'df': hate_dataset,
        'prefix': 'hate',
        'name': 'Hate Speech',
        'description': 'Dataset of hate speech annotated on Internet forum posts in English at sentence-level. The source forum in Stormfront, a large online community of white nacionalists. A total of 10,568 sentence have been been extracted from Stormfront and classified as conveying hate speech or not.',
        'text_column': 'text',
        'target_column': 'label',
        'target': 'label (0/1), 1 is hate, 0 is noHate',  
    },
]

## LLM-based Feature Discovery

In [16]:
from langchain_openai import ChatOpenAI

# Inicializace ChatGPT
llm = ChatOpenAI(
    model_name="gpt-4o-2024-08-06", 
    temperature=0,
    max_tokens=4096,
    top_p=0.9,  # Omezuje pravděpodobnostní prostor odpovědí
    frequency_penalty=0,  # Minimalizuje variabilitu ve výstupech
    presence_penalty=0,  # Zajišťuje, že odpovědi nebudou příliš rozmanité
    api_key="<key>"
)

response = llm.invoke('Who are you?')
response.content

'I am an AI language model created by OpenAI, designed to assist with a wide range of questions and tasks by providing information and generating text based on the input I receive. How can I assist you today?'

In [17]:
prompts = {}
for i, dataset in enumerate(datasets):
    response = llm.invoke(generate_prompt(dataset['name'], dataset['description'], dataset['target'], dataset['df'], dataset['text_column'], dataset['target_column']))
    prompts[i] = response.content
# prompts

In [18]:
import re
import json

def extract_json(response: str):
    # Find JSON between ```json and ```
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            raise ValueError(f"Chyba při dekódování JSON: {e}")
    else:
        return json.loads(response.strip('```json').strip('```'))

import json
def clean_json_output(response_dict):
    cleaned_dict = {}
    for key, value in response_dict.items():
        value = extract_json(value)
        cleaned_dict[key] = value
    return cleaned_dict

In [19]:
# Remove markdown
clean_response = clean_json_output(prompts)

clean_response

{0: {'features': [{'feature_name': 'intent_category',
    'description': 'The specific intent category of the customer query within the banking domain.',
    'possible_values': ['activate_my_card',
     'age_limit',
     'apple_pay_or_google_pay',
     'atm_support',
     'automatic_top_up',
     'balance_not_updated_after_bank_transfer',
     'balance_not_updated_after_cheque_or_cash_deposit',
     'beneficiary_not_allowed',
     'cancel_transfer',
     'card_about_to_expire',
     'card_acceptance',
     'card_arrival',
     'card_delivery_estimate',
     'card_linking',
     'card_not_working',
     'card_payment_fee_charged',
     'card_payment_not_recognised',
     'card_payment_wrong_exchange_rate',
     'card_swallowed',
     'cash_withdrawal_charge',
     'cash_withdrawal_not_recognised',
     'change_pin',
     'compromised_card',
     'contactless_not_working',
     'country_support',
     'declined_card_payment',
     'declined_cash_withdrawal',
     'declined_transfer',
   

## Feature Generation Prompt

In [20]:
def generate_feature_queries(features, text):
    queries = {
        "input_text": text,
        "task": "Extract the following features as described below and return a valid JSON object.",
        "constraints": [
            "The output must be a valid JSON.",
            "All answers must be simple and correspond to categorical values only."
        ],
        "features": [
            {
                "feature_name": feature['feature_name'],
                "description": feature['description'],
                "extraction_query": feature['extraction_query'].replace('{text}', text)
            } for feature in features['features']
        ],
        "output_format": {
            "type": "json",
            "structure": {
                "features": [
                    {
                        "feature_name": "<Feature Name>",
                        "answer": "<Extracted Answer>"
                    }
                ]
            }
        }
    }
    return json.dumps(queries)

### Feature Generation with Batch API

In [21]:
# Create a JSONL file for the Batch API
def create_batch_file(df, text_column, file, features):
    if os.path.exists(file):
        print(f"The file '{file}' already exists.")
        return file
    with open(file, "w", encoding="utf-8") as f:
        for rid, row in df.iterrows():
            text = row[text_column]
            query = generate_feature_queries(features, text)
    
            request_data = {
                "custom_id": f"{rid}", # Unique ID for each request
                "method": "POST",
                "url": "/v1/chat/completions", # Endpoint that the Batch API will process
                "body": {
                    "model": "gpt-4o-mini-2024-07-18", # Or another supported model
                    "temperature": 0,
                    "max_tokens": 4096,
                    "top_p": 0.9,  # Restricts the probability space of responses
                    "frequency_penalty": 0,  # Minimizes output variability
                    "presence_penalty": 0,  
                    "messages": [
                        {"role": "system", "content": "You are an expert in extracting categorical features from text. IMPORTANT: Return only a valid JSON object with no aditional explanations, text, or markdown. Do not use triple backticks, and do not include any commentary or introductory text."},
                        {"role": "user", "content": query}
                    ]
                }
            }
            # Zapsat každou žádost na jeden řádek JSONL
            f.write(json.dumps(request_data, ensure_ascii=False) + "\n")
    print("\nBatch created.")
    print(f"Generated .jsonl contains {len(df)} requests.")
    return file

In [22]:
for i, dataset in enumerate(datasets):
    df = dataset['df']
    text_column = dataset['text_column']
    file = "input_" + dataset['prefix'] + "_batch.jsonl"
    features = clean_response[i]
    batch_file = create_batch_file(df, text_column, file, features)
    dataset['batch_file'] = batch_file
datasets    

The file 'input_bank77_batch.jsonl' already exists.
The file 'input_hate_batch.jsonl' already exists.
The file 'input_fake_batch.jsonl' already exists.
The file 'input_math_batch.jsonl' already exists.
The file 'input_bank_batch.jsonl' already exists.


[{'df':                                                    text  label
  0                        I am still waiting on my card?     11
  1     What can I do if my card still hasn't arrived ...     11
  2     I have been waiting over a week. Is the card s...     11
  3     Can I track my card while it is in the process...     11
  4     How do I know if I will get my card, or if it ...     11
  ...                                                 ...    ...
  3075      If i'm not in the UK, can I still get a card?     24
  3076                 How many countries do you support?     24
  3077              What countries do you do business in?     24
  3078             What are the countries you operate in.     24
  3079         Can the card be mailed and used in Europe?     24
  
  [13083 rows x 2 columns],
  'prefix': 'bank77',
  'name': 'BANKING77 dataset',
  'description': 'BANKING77 dataset provides a very fine-grained set of intents in a banking domain. It comprises 13,083 customer 

### API Calls

In [23]:
import os
import tiktoken

def split_file_by_token(input_file, max_tokens=2000, encoding_name="gpt2",
                        output_prefix="batch_part", output_extension=".jsonl"):
    """
    Splits an input file into multiple chunks such that each chunk does not exceed max_tokens.
    Tokens are counted using the specified encoding from the tiktoken library.
    
    Parameters:
      input_file (str): Path to the input file.
      max_tokens (int): Maximum number of tokens allowed per chunk.
      encoding_name (str): Name of the token encoding to use (default "gpt2").
      output_prefix (str): Prefix for the output file names.
      output_extension (str): Extension for the output files.
    
    Returns:
      List[str]: A list of created output file paths.
    """
    # Get the encoding for token counting.
    encoding = tiktoken.get_encoding(encoding_name)
    
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    chunks = []
    current_chunk = []
    current_tokens = 0
    all_tokens = 0
    
    # Process each line and accumulate tokens until the limit is reached.
    for line in lines:
        line_token_count = len(encoding.encode(line))
        if current_tokens + line_token_count > max_tokens and current_chunk:
            # If adding this line would exceed the limit, save the current chunk.
            chunks.append("".join(current_chunk))
            current_chunk = [line]
            current_tokens = line_token_count
        else:
            current_chunk.append(line)
            current_tokens += line_token_count
            
    if current_chunk:
        chunks.append("".join(current_chunk))
    
    output_files = []
    # Write each chunk to a separate output file.
    for idx, chunk in enumerate(chunks, start=1):
        output_filename = f"{output_prefix}{idx}{output_extension}"
        if os.path.exists(output_filename):
            print(f"The file '{output_filename}' already exists.")
            output_files.append(output_filename)
            continue
        with open(output_filename, "w", encoding="utf-8") as out_f:
            out_f.write(chunk)
        output_files.append(output_filename)
        token_count = len(encoding.encode(chunk))
        print(f"{output_filename}: {token_count} tokens")
        all_tokens += token_count
    
    print("\nDone! Created files:")
    for file in output_files:
        print(f" - {file}")
    print("Total tokens:")
    print(all_tokens)
    return output_files

In [24]:
# Upload batch file via File API
def upload_file(file, api_key):
    with open(file, "rb") as file_data:
        response = requests.post(
            "https://api.openai.com/v1/files",
            headers={"Authorization": f"Bearer {api_key}"},
            files={
                "file": ("batch_input.jsonl", file_data, "application/jsonl")
            },
            data={"purpose": "batch"}
        )
    print("\nUpload file:")
    print(file)
    print(response.json()['status'])
    return response.json()['id']

In [25]:
# Start batch processing
def start_batch(file_id, api_key):
    batch_payload = {
        "input_file_id": file_id,
        "endpoint": "/v1/chat/completions",
        "completion_window": "24h",  # only allowed value for 50% discount
    }
    response = requests.post(
        "https://api.openai.com/v1/batches",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json=batch_payload
    )
    batch_info = response.json()
    print("\nStart batch processing:")
    print(response.json()['status'])
    return response.json()['id']

In [26]:
# Batch processing status
def check_status(batch_id, api_key):
    check_status = requests.get(
        f"https://api.openai.com/v1/batches/{batch_id}",
        headers={"Authorization": f"Bearer {api_key}"}
    )
    print("\nCheck status:")
    print(check_status.json()['status'])
    print("Errors:")
    print(check_status.json()['errors'])
    return check_status.json()['output_file_id'], check_status.json()['status']

In [27]:
# Download the content of the output file
def download_processed_file(output_file_id, output_file, api_key):
    response = requests.get(
        f"https://api.openai.com/v1/files/{output_file_id}/content",
        headers={"Authorization": f"Bearer {api_key}"}
    )
    
    with open(output_file, "wb") as f:
        f.write(response.content)
    print("\nBatch results written to:")
    print(output_file)
    return output_file

### Run Batch API Workflow

In [28]:
api_key = "<key>"

In [29]:
def process_dataset(dataset, api_key):
    """
    Process one dataset: split the batch file into chunks, upload each chunk,
    start batch processing, check status (with retries) and finally download
    the processed file.
    """
    prefix = dataset['prefix']
    batch_file = dataset['batch_file']
    
    # Inform the user about the dataset
    print(f"\nProcessing dataset: {dataset['name']}")
    print(f"Description: {dataset['description']}")
    
    # Split the batch file into chunks (using max_tokens=1800000)
    output_prefix = f"{prefix}_batch_part"
    print("Splitting batch file...")
    chunk_files = split_file_by_token(
        input_file=batch_file,
        max_tokens=1800000,
        encoding_name="gpt2",
        output_prefix=output_prefix,
        output_extension=".jsonl"
    )
    
    # Process each chunk file
    for i, chunk in enumerate(chunk_files, start=1):
        output_file_name = f"{prefix}_output_chunk{i}.txt"
        if os.path.exists(output_file_name):
            print(f"The file '{output_file_name}' already exists.")
            continue
        print(f"\nProcessing chunk {i} of {len(chunk_files)}: {chunk}")
        file_id = upload_file(chunk, api_key)
        print(f"Uploaded file. Received file_id: {file_id}")
        
        max_retries = 24
        retry_count = 0
        batch_completed = False
        
        # Retry loop for starting the batch and checking its status
        while retry_count < max_retries and not batch_completed:
            batch_id = start_batch(file_id, api_key)
            print(f"Started batch. Received batch_id: {batch_id}")
            
            status_retry = 0
            while status_retry < max_retries:
                output_file_id, status = check_status(batch_id, api_key)
                print(f"Batch status: {status}")
                
                # If completed, break out of the inner loop
                if status.lower() == "completed":
                    batch_completed = True
                    break
                # If status indicates failure or cancellation, break to retry batch start
                elif status.lower() in ["failed", "cancelled"]:
                    print(f"Batch {batch_id} {status}. Retrying batch start in 1 hour...")
                    time.sleep(3600)  # Wait 1 hour
                    break
                else:
                    # Status is not completed (but not failed/cancelled): wait 1 hour and check again
                    print(f"Batch {batch_id} not completed yet. Checking again in 1 hour...")
                    time.sleep(3600)
                    status_retry += 1
            
            # If after inner loop batch is completed, proceed; otherwise, increment retry_count and retry start_batch
            if not batch_completed:
                retry_count += 1
                print(f"Retrying start_batch for chunk {i}. Attempt {retry_count}/{max_retries}...")
        
        if not batch_completed:
            print(f"Error: Maximum retries reached for chunk {i}. Aborting processing for dataset {dataset['name']}.")
            return False
        
        # Batch job completed: download the processed output file.
        downloaded_file = download_processed_file(output_file_id, output_file_name, api_key)
        print(f"Downloaded processed file: {downloaded_file}")
    
    return True

In [30]:
# List of dataset dictionaries to process
def run(datasets, api_key):
    overall_start = datetime.datetime.now()
    print("Process started at:", overall_start.strftime("%Y-%m-%d %H:%M:%S"))
    
    # Iterate over each dataset
    for dataset in datasets:
        dataset_start = datetime.datetime.now()
        print("\n============================")
        print(f"Starting processing for dataset: {dataset['name']}")
        print("Start time:", dataset_start.strftime("%Y-%m-%d %H:%M:%S"))
        
        success = process_dataset(dataset, api_key)
        if not success:
            print(f"Processing failed for dataset: {dataset['name']}. Aborting overall process.")
            return
        
        dataset_end = datetime.datetime.now()
        elapsed = dataset_end - dataset_start
        print(f"Finished processing dataset: {dataset['name']}")
        print("End time:", dataset_end.strftime("%Y-%m-%d %H:%M:%S"))
        print("Elapsed time for this dataset:", str(elapsed))
        print("============================\n")
    
    overall_end = datetime.datetime.now()
    total_elapsed = overall_end - overall_start
    print("All datasets processed.")
    print("Process ended at:", overall_end.strftime("%Y-%m-%d %H:%M:%S"))
    print("Total elapsed time:", str(total_elapsed))

In [ ]:
run(datasets, api_key)

Process started at: 2025-03-11 16:49:45

Starting processing for dataset: BANKING77 dataset
Start time: 2025-03-11 16:49:45

Processing dataset: BANKING77 dataset
Description: BANKING77 dataset provides a very fine-grained set of intents in a banking domain. It comprises 13,083 customer service queries labeled with 77 intents. It focuses on fine-grained single-domain intent detection.
Splitting batch file...
The file 'bank77_batch_part1.jsonl' already exists.
The file 'bank77_batch_part2.jsonl' already exists.
The file 'bank77_batch_part3.jsonl' already exists.
The file 'bank77_batch_part4.jsonl' already exists.
The file 'bank77_batch_part5.jsonl' already exists.
The file 'bank77_batch_part6.jsonl' already exists.
The file 'bank77_batch_part7.jsonl' already exists.
The file 'bank77_batch_part8.jsonl' already exists.
The file 'bank77_batch_part9.jsonl' already exists.
The file 'bank77_batch_part10.jsonl' already exists.
The file 'bank77_batch_part11.jsonl' already exists.
The file 'bank